# Multi-phase model heatmaps for the web demo\n\n> **Research Use Only.** This notebook exports only offline sensitivity artefacts. A heatmap is not a lesion segmentation and has no diagnostic meaning.\n\nRun the E4 cache/checkpoint bootstrap from the evaluation notebook first. For each demo patient, use the checkpoint from the fold where that patient is validation data. The code below deliberately calls the preprocessing helper instead of rebuilding crop geometry in the notebook.

In [ ]:
# Required variables from the E4 bootstrap: REPO, model, DEVICE, pid, fold, and CACHE.\n# `model` is the OOF fold model in eval mode; cache contains the exact E4 input.\nimport numpy as np\nimport torch\nfrom pathlib import Path\n\nfrom src.data.annotation import Annotation\nfrom src.data.images import DEFAULT_LABEL_SUFFIXES, scan_image_index\nfrom src.preprocess.build_cache import resample_annotation_masks\nfrom src.utils.io import load_yaml, resolve_data_root\nfrom src.xai.input_heatmap import input_x_gradient_heatmaps\n\nDATA_CFG = load_yaml(REPO / 'configs' / 'data.yaml')\nPREPROCESS_CFG = load_yaml(REPO / 'configs' / 'preprocess_e4.yaml')\nDATA_ROOT = resolve_data_root(DATA_CFG)\nannotation = Annotation(DATA_ROOT / DATA_CFG['annotation_rel'])\nimage_index = scan_image_index(DATA_ROOT / DATA_CFG['images_rel'], DATA_CFG['image_suffixes'])\nmask_index = scan_image_index(\n    DATA_ROOT / DATA_CFG['labels_rel'], DATA_CFG.get('label_suffixes', DEFAULT_LABEL_SUFFIXES)\n)\nPHASE_TOKENS = [phase['file'] for phase in DATA_CFG['phases']]\n\nwith np.load(next(CACHE.glob(f'{pid}*.npz'))) as cached:\n    crop = np.asarray(cached['image'], dtype=np.float32)\nvolume = torch.from_numpy(crop)[None].to(DEVICE)\nmodel.eval()\nwith torch.no_grad():\n    pred = int(model(volume).argmax(dim=1).item())\nresult = input_x_gradient_heatmaps(model, volume, pred)\nmasks = resample_annotation_masks(\n    pid, annotation, image_index, DATA_CFG['phases'], PREPROCESS_CFG, mask_index\n)\nassert tuple(crop.shape) == tuple(result.heatmaps.shape) == tuple(masks.shape)\nassert len(PHASE_TOKENS) == crop.shape[0] == 8\n\n# Display copies only: do not replace model inputs or the shared heatmap scale.\nlo = crop.min(axis=(1, 2, 3), keepdims=True)\nhi = crop.max(axis=(1, 2, 3), keepdims=True)\ncrop_refs = np.clip((crop - lo) / np.maximum(hi - lo, 1e-6) * 255, 0, 255).astype(np.uint8)\nOUT = Path('/kaggle/working/model_heatmaps')\nOUT.mkdir(parents=True, exist_ok=True)\nnp.savez_compressed(\n    OUT / f'{pid}.npz',\n    phase_tokens=np.asarray(PHASE_TOKENS),\n    crop_refs=crop_refs,\n    heatmaps_pred=result.heatmaps.numpy().astype(np.float16),\n    annotation_masks=masks.astype(np.uint8),\n    pred_index=np.int64(pred),\n    heatmap_scale=np.float32(result.scale),\n)\nprint(f'{pid}: fold {fold}, predicted class {pred}, global sensitivity scale {result.scale:.5g}')

Copy the resulting `.npz` files to `runs/E4_per_phase_results/model_heatmaps/` on the demo machine, or set `LLDMMRI_MODEL_HEATMAP_DIR`. The web app rejects an artefact if phase order or any image/heatmap/mask shape differs.